# Assignment 4.1: Layered, feedforward neural networks

The networks in this notebook are built from PyTorch layers, and PyTorch computes every gradient
for you, so you do not implement backpropagation. The training function, the accuracy function,
the cross-validation, and the plots are provided. Your job is to build the three network
architectures from the spec, choose the number of training epochs for each dataset, and write the
report.

Every model is evaluated with 5-fold cross-validation. Each dataset is split into five folds; the
model is trained five times, each time on four folds, and tested on the fold it did not see. The
results are the averages over the five runs, so they depend much less on one lucky or unlucky split.

Four code cells are yours: `build_model_1`, `build_model_2`, `build_model_3`, and the settings
cell. Everything else is provided and already runs.

# Running in Colab

Open this notebook from its Colab link in the spec, then save your own copy right away with
**File > Save a copy in Drive**. Work in that copy, or your changes are lost when Colab closes.
This notebook does not need a GPU.

The next cell copies the course repository onto the Colab machine so the notebook finds its data in
`../data`. Run it first every time you open the notebook. On your own computer it does nothing.

In [ ]:
import os, sys, subprocess

# in colab, fetch the course repository so ../data exists
if "google.colab" in sys.modules:
    if not os.path.exists("/content/fund-ml-project-4"):
        subprocess.run(["git", "clone", "--depth", "1", "https://github.com/zacharytgray/fund-ml-project-4",
                        "/content/fund-ml-project-4"], check=True)
    os.chdir("/content/fund-ml-project-4/assignment")

# Import libraries
Do not use any other Python library.

numpy - Linear algebra library for handling vectors and matrices, collectively processed as numpy arrays.

matplotlib - Graphing library for visualizing results.

csv - Library used for reading csv files into python lists.

sklearn - Machine learning library from which we source the digits dataset and the cross-validation folds.

torch - PyTorch, the neural network library. `torch.nn` holds the layers and the loss function, and `torch.optim` holds the optimizer.

time - Simple library for timing code.

os, sys, subprocess - Used only by the Colab setup cell above.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from csv import reader
from sklearn.datasets import load_digits
from sklearn.model_selection import StratifiedKFold
import torch
import torch.nn as nn
from time import time

# Load Datasets

These functions load the three datasets.

`load_digits_data` returns the 1797 handwritten digits, each one 64 pixels with an intensity from
0 to 16. There are ten classes.

`load_credit` returns the 1000 loan applications of the Statlog German Credit dataset from
`../data/credit.csv`. Each application has 20 numeric features, and the target is 1 for a
creditworthy applicant and 0 otherwise.

`load_mushroom` returns the 8124 mushrooms of the UCI Mushroom dataset from `../data/mushroom.csv`.
Of its 22 categorical features, veil type has a single value and is dropped. The other 21 are
one-hot encoded into 116 columns of 0s and 1s, with a missing stalk root kept as its own category. The target is 1 for a poisonous mushroom and 0 for an
edible one.

Outputs:

*   **features**: numpy array of shape (N, number of features)
*   **labels**: numpy array of N integer class labels

# Function: make_folds

Splits one dataset into five folds for cross-validation and returns the five (training indices, test
indices) pairs. Each fold is the test set once, and the other four are its training set. The folds
are stratified, so each keeps the class proportions of the full dataset, and seeded, so every run
gets the same folds.

# Function: standardize

Rescales every feature to mean 0 and standard deviation 1, using the mean and standard deviation of
the training set for both sets, so nothing about the test fold leaks into training. A feature that
is constant in the training set is only centered.
One German Credit feature is a loan amount in the thousands, and without scaling it dominates
the first epochs of gradient descent.

In [ ]:
def load_digits_data(plot=True):
    # optical digits dataset from scikit-learn
    digits = load_digits()
    features = digits.data
    labels = digits.target

    if plot:
        images = features.reshape(-1, 8, 8)
        plt.figure(figsize=(10, 5))
        for i in range(10):
            plt.subplot(3, 4, i + 1)
            plt.imshow(images[i], cmap='gray')
            plt.title(f'Handwritten {labels[i]}')
            plt.axis('off')
        plt.tight_layout()
        plt.show()

    return features, labels


def load_csv(path):
    # numeric csv with a header row, last column is the class
    with open(path, 'r') as file:
        rows = list(reader(file))[1:]
    features = np.array([[float(value) for value in row[:-1]] for row in rows])
    labels = np.array([int(row[-1]) for row in rows])
    return features, labels


def plot_class_counts(labels, title):
    plt.figure(figsize=(5, 4))
    plt.bar(['0', '1'], [np.sum(labels == 0), np.sum(labels == 1)])
    plt.xlabel('Class')
    plt.ylabel('Instances')
    plt.title(title)
    plt.show()


def load_credit(plot=True):
    features, labels = load_csv('../data/credit.csv')
    if plot:
        plot_class_counts(labels, 'German Credit class counts')
    return features, labels


def load_mushroom(plot=True):
    features, labels = load_csv('../data/mushroom.csv')
    if plot:
        plot_class_counts(labels, 'Mushroom class counts')
    return features, labels


def make_folds(features, labels):
    folds = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=SEED)
    return list(folds.split(features, labels))


def standardize(X_train, X_test):
    mean = X_train.mean(axis=0)
    std = X_train.std(axis=0)
    std[std == 0] = 1.0
    return (X_train - mean) / std, (X_test - mean) / std

# Training settings

The spec fixes the optimizer for every model in this notebook: stochastic gradient descent with
learning rate $\eta = 0.01$ and L2 regularization strength $\lambda = 0.001$. PyTorch's `SGD` takes
the L2 strength as its `weight_decay` argument. Each step uses a minibatch of `BATCH_SIZE` training
instances, and every network starts from the same seeded initial weights. `N_FOLDS` is the number
of cross-validation folds.

In [ ]:
LEARNING_RATE = 0.01
WEIGHT_DECAY = 0.001
BATCH_SIZE = 32
N_FOLDS = 5
SEED = 0

# Models

A PyTorch network is a stack of layers. You will use two kinds.

[Linear layers](https://docs.pytorch.org/docs/stable/generated/torch.nn.Linear.html) (also known as
dense layers) hold a matrix of weights and a bias vector. `nn.Linear(a, b)` takes a vector of `a`
values and returns a vector of `b` values.

[ReLU layers](https://docs.pytorch.org/docs/stable/generated/torch.nn.ReLU.html) apply the ReLU
function to each value. They have no weights.

Chain the layers with a [Sequential model](https://docs.pytorch.org/docs/stable/generated/torch.nn.Sequential.html).
The last layer of every model is a linear layer with one output per class and no activation. The
softmax that turns those outputs into class probabilities is part of the cross-entropy loss, which
the training function applies.

# Function: build_model_1

The small network: one hidden layer of 16 units.

1.  Linear(input count, 16) + ReLU
2.  Linear(16, class count)

Inputs:
*   **n_inputs**: number of input features (integer)
*   **n_classes**: number of classes (integer)

Output:
*   **model**: an `nn.Sequential` network

In [ ]:
def build_model_1(n_inputs, n_classes):
    # 1. stack the hidden linear layer, its relu, and the output layer in a sequential model
    # TODO
    pass

# Function: build_model_2

The deep network: four hidden layers of 100 units, five linear layers in all.

1.  Linear(input count, 100) + ReLU
2.  Linear(100, 100) + ReLU
3.  Linear(100, 100) + ReLU
4.  Linear(100, 100) + ReLU
5.  Linear(100, class count)

Inputs:
*   **n_inputs**: number of input features (integer)
*   **n_classes**: number of classes (integer)

Output:
*   **model**: an `nn.Sequential` network

In [ ]:
def build_model_2(n_inputs, n_classes):
    # 1. stack the four hidden linear layers, each followed by a relu, then the output layer
    # TODO
    pass

# Function: build_model_3

The wide network: the same shape as Model 1, but the hidden layer has `hidden_width` units. That
width, L, is given for each dataset below.

1.  Linear(input count, L) + ReLU
2.  Linear(L, class count)

Inputs:
*   **n_inputs**: number of input features (integer)
*   **n_classes**: number of classes (integer)
*   **hidden_width**: number of hidden units L (integer)

Output:
*   **model**: an `nn.Sequential` network

In [ ]:
def build_model_3(n_inputs, n_classes, hidden_width):
    # 1. the same stack as model 1, with hidden_width units in the hidden layer
    # TODO
    pass

# Function: count_parameters

This function is provided. It returns the number of trainable values in a network: every weight and
every bias of every layer.

# Model 3 width

The width L of Model 3 is chosen so that Model 3 has about as many parameters as Model 2. A linear
layer from a inputs to b outputs has (a + 1) b parameters: one weight per input for each output, plus
a bias. With n inputs and k classes, Model 3 therefore has (n + 1) L + (L + 1) k parameters. Setting
that equal to Model 2's count and solving for L gives the values in `MODEL_3_WIDTH`. L depends on n
and k, so it differs between datasets.

The next cell prints the input count, the class count, L, and the parameter counts of all three
models for each dataset.

In [ ]:
def count_parameters(model):
    return sum(p.numel() for p in model.parameters() if p.requires_grad)


MODEL_3_WIDTH = {"digits": 504, "credit": 1417, "mushroom": 355}

DATASETS = {
    "digits": load_digits_data,
    "credit": load_credit,
    "mushroom": load_mushroom,
}

print(f"{'dataset':10} {'inputs':>7} {'classes':>8} {'L':>6} {'model 1':>9} {'model 2':>9} {'model 3':>9}")
for dataset_name, loader in DATASETS.items():
    features, labels = loader(plot=False)
    n_inputs, n_classes = features.shape[1], len(np.unique(labels))
    params_1 = count_parameters(build_model_1(n_inputs, n_classes))
    params_2 = count_parameters(build_model_2(n_inputs, n_classes))
    params_3 = count_parameters(build_model_3(n_inputs, n_classes, MODEL_3_WIDTH[dataset_name]))
    print(f"{dataset_name:10} {n_inputs:7d} {n_classes:8d} {MODEL_3_WIDTH[dataset_name]:6d} "
          f"{params_1:9d} {params_2:9d} {params_3:9d}")

# Settings

`EPOCHS` is the number of passes over the training set, one value for each dataset. Train each model to convergence: run long
enough that the training loss has flattened out in the loss plots below. Start with a generous
value, read the plots, and adjust. All three models on a dataset use the same number of epochs.

In [ ]:
# 1. for each dataset, the number of epochs that trains the models to convergence
# TODO
EPOCHS = {"digits": None, "credit": None, "mushroom": None}

# Function: train_model

This function is provided. It trains one network with minibatch stochastic gradient descent on the
cross-entropy loss, using the settings above. After every epoch it computes the cross-entropy loss
of the whole training set and of the whole test set, with the network in evaluation mode. The test
loss is only recorded for the plots; it never changes the weights. In cross-validation the test
set is the held-out fold.

Inputs:
*   **model**: a network from one of your build functions
*   **X_train**, **y_train**: training features and labels as tensors
*   **X_test**, **y_test**: test features and labels as tensors
*   **epochs**: number of passes over the training set (integer)

Output:
*   **history**: dictionary with lists **train_loss** and **test_loss**, one value per epoch

# Function: accuracy

This function is provided. It returns the percentage of instances whose highest-scoring class is the
true class.

In [ ]:
def train_model(model, X_train, y_train, X_test, y_test, epochs):
    loss_fn = nn.CrossEntropyLoss()
    optimizer = torch.optim.SGD(model.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY)
    generator = torch.Generator().manual_seed(SEED)
    history = {"train_loss": [], "test_loss": []}
    for epoch in range(epochs):
        model.train()
        order = torch.randperm(len(X_train), generator=generator)
        for start in range(0, len(X_train), BATCH_SIZE):
            batch = order[start:start + BATCH_SIZE]
            optimizer.zero_grad()
            loss = loss_fn(model(X_train[batch]), y_train[batch])
            loss.backward()
            optimizer.step()
        model.eval()
        with torch.no_grad():
            history["train_loss"].append(loss_fn(model(X_train), y_train).item())
            history["test_loss"].append(loss_fn(model(X_test), y_test).item())
    return history


def accuracy(model, X, y):
    model.eval()
    with torch.no_grad():
        predictions = model(X).argmax(dim=1)
    return (predictions == y).float().mean().item() * 100

# Main code

Loads each dataset and splits it into five folds. For each model and each fold, it standardizes the
features, trains the model from the same seed on the other four folds, and tests it on that fold.
Each plot shows the training and test loss for every epoch, averaged over the five folds, with the
test loss of each single fold drawn faintly behind. For German Credit it also prints the share of
the majority class, which is the accuracy of a model that always predicts that class.

In [ ]:
results = []
for dataset_name, loader in DATASETS.items():
    features, labels = loader(plot=True)
    n_inputs, n_classes = features.shape[1], len(np.unique(labels))
    folds = make_folds(features, labels)

    if dataset_name == "credit":
        majority = max(np.mean(labels == 0), np.mean(labels == 1)) * 100
        print(f"credit: majority class share {majority:.2f}%")

    builders = {
        "Model 1": lambda: build_model_1(n_inputs, n_classes),
        "Model 2": lambda: build_model_2(n_inputs, n_classes),
        "Model 3": lambda: build_model_3(n_inputs, n_classes, MODEL_3_WIDTH[dataset_name]),
    }
    fig, axes = plt.subplots(1, 3, figsize=(15, 4))
    for ax, (model_name, build) in zip(axes, builders.items()):
        histories, train_accuracies, test_accuracies = [], [], []
        start = time()
        for train_index, test_index in folds:
            X_train, X_test = standardize(features[train_index], features[test_index])
            X_train = torch.tensor(X_train, dtype=torch.float32)
            X_test = torch.tensor(X_test, dtype=torch.float32)
            y_train = torch.tensor(labels[train_index], dtype=torch.long)
            y_test = torch.tensor(labels[test_index], dtype=torch.long)
            torch.manual_seed(SEED)
            model = build()
            histories.append(train_model(model, X_train, y_train, X_test, y_test, EPOCHS[dataset_name]))
            train_accuracies.append(accuracy(model, X_train, y_train))
            test_accuracies.append(accuracy(model, X_test, y_test))
        seconds = time() - start

        train_loss = np.mean([h["train_loss"] for h in histories], axis=0)
        test_loss = np.mean([h["test_loss"] for h in histories], axis=0)
        results.append({
            "dataset": dataset_name, "model": model_name,
            "parameters": count_parameters(model), "epochs": EPOCHS[dataset_name],
            "train_loss": train_loss[-1], "test_loss": test_loss[-1],
            "train_accuracy": np.mean(train_accuracies),
            "test_accuracy": np.mean(test_accuracies), "test_std": np.std(test_accuracies),
            "fold_accuracies": test_accuracies, "seconds": seconds,
        })
        epochs = range(1, len(train_loss) + 1)
        for h in histories:
            ax.plot(epochs, h["test_loss"], color="tab:orange", alpha=0.25, linewidth=0.8)
        ax.plot(epochs, train_loss, color="tab:blue", label="train loss (mean of folds)")
        ax.plot(epochs, test_loss, color="tab:orange", label="test loss (mean of folds)")
        ax.set_xlabel("epoch")
        ax.set_ylabel("cross-entropy loss")
        ax.set_title(f"{model_name} on {dataset_name}")
        ax.legend()
    plt.tight_layout()
    plt.show()

# Results table

One row per model: its parameter count, the epochs it trained for, and its final training and test
loss and accuracy, each averaged over the five folds. `test std` is the standard deviation of the
test accuracy across the folds, and the second table lists the test accuracy of every fold.

In [ ]:
print(f"{'dataset':10} {'model':8} {'params':>8} {'epochs':>7} {'train loss':>11} {'test loss':>10} "
      f"{'train acc':>10} {'test acc':>9} {'test std':>9} {'seconds':>8}")
for r in results:
    print(f"{r['dataset']:10} {r['model']:8} {r['parameters']:8d} {r['epochs']:7d} "
          f"{r['train_loss']:11.4f} {r['test_loss']:10.4f} {r['train_accuracy']:9.2f}% "
          f"{r['test_accuracy']:8.2f}% {r['test_std']:8.2f}% {r['seconds']:8.1f}")

print()
print(f"{'dataset':10} {'model':8} " + " ".join(f"{'fold ' + str(i + 1):>8}" for i in range(N_FOLDS)))
for r in results:
    print(f"{r['dataset']:10} {r['model']:8} " + " ".join(f"{a:7.2f}%" for a in r["fold_accuracies"]))

# Report

Cover the following in your report.

*   The parameter counts of the three models on each dataset, and the number of epochs you chose for each dataset and why.
*   The loss plot of each of the nine models.
*   Each model's mean test accuracy over the five folds, with its standard deviation.
*   For each dataset, compare the deep network (Model 2) with the wide network (Model 3), say whether the differences between models are larger than the spread across folds, and note where a model's test loss rises while its training loss keeps falling.